# 9. Bias in the data: Titanic survival

BDI's `ebdai` package looks at **bias in the labels** — the association between a sensitive attribute and the outcome that is already in the table, before any model exists — and, once a fuzzy rule model is fitted, at **bias in inference**: which rules fire for whom, and how predictions and errors are distributed across groups.

This notebook uses the Titanic table from the [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025) workshop (Raquel Fernandez Peralta and Javier Fumanal Idocin), originally written for Ex-Fuzzy 2.1.3. The current `ex_fuzzy` API still trains the classifier; the new helpers live in `ebdai`.

`import ex_fuzzy` is the rule learner. `import ebdai` is the bias layer.

The route through the notebook:

| Step | Question | Call |
| --- | --- | --- |
| 1 | How often did the outcome happen in each group, before any model? | `outcome_rates_by_group` |
| 2 | What sentences does a rule search write on these data? | `BaseFuzzyRulesClassifier.fit`, `eval_tools.eval_fuzzy_model` |
| 3 | How often is the outcome *predicted* in each group, and where do the errors fall? | `fairness_report` |
| 4 | Which rule decided each prediction, and for whom? | `winning_rules_by_group` |

Nothing here mitigates anything — demo 11 does that. This notebook is about seeing bias, in a model family where seeing it means reading sentences instead of probing a black box.

On [Google Colab](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/09_bias_titanic.ipynb), run the setup cell below first. It follows the workshop install guide: clone this repository, move into `EBD-AI`, and `pip install` it. The workshop table ships with that install. If the next cell cannot import `ebdai` just after the install, restart the runtime and run every cell again.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/09_bias_titanic.ipynb)

## Setup (Google Colab only)

The next cell clones this repository and installs it, exactly as the workshop install guide does. It is the only cell that is not part of the analysis: skip it if you already have the package (`pip install ebdai`, or `pip install -e .` from a checkout). On Colab it takes a minute, and the workshop tables arrive with the install.

In [ ]:
!git clone -q https://github.com/HAISymbiosis/EBD-AI.git
%cd EBD-AI
!pip install -q .


## The table

`load_titanic` returns the workshop table already prepared, together with the name of the sensitive column:

- the identifier and free-text columns (`ID`, `Name`, `Ticket`, `Cabin`) are dropped;
- rows with a missing `Age` or `Embarked` are dropped, leaving 712 of the 891 passengers. `ex_fuzzy` rejects missing values rather than imputing them silently, so the loader removes them up front. Demo 02 shows the other route — an imputation step inside a scikit-learn `Pipeline`.

The remaining columns: `Class` is the ticket class (1–3), `SibSp` counts siblings and spouses aboard, `Parch` counts parents and children aboard, `Fare` is the ticket price, and `Embarked` is the port of embarkation (`C` Cherbourg, `Q` Queenstown, `S` Southampton). The label is `Survived` (1 survived, 0 died); the sensitive attribute is `Sex`.

`features_and_target` splits the frame into `X` (every other column) and `y` (the label). Note that `Sex` **stays inside `X`**: the model is allowed to use it, which is what makes the rule printout later in the notebook worth reading.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS
from ex_fuzzy import eval_tools

from ebdai import (
    features_and_target,
    load_titanic,
    outcome_rates_by_group,
    plot_outcome_rates,
    plot_winning_rules_by_group,
    fairness_report,
    parse_printed_rules,
    winning_rules_by_group,
)

frame, sensitive = load_titanic()
X, y = features_and_target(frame, 'Survived')
print(X.head())
print('rows', len(X), 'sensitive', sensitive)

   Class     Sex   Age  SibSp  Parch     Fare Embarked
0      3    male  22.0      1      0   7.2500        S
1      1  female  38.0      1      0  71.2833        C
2      3  female  26.0      0      0   7.9250        S
3      1  female  35.0      1      0  53.1000        S
4      3    male  35.0      0      0   8.0500        S
rows 712 sensitive Sex


## Step 1. Bias in the data: the outcome rate by group

The first measurement needs no model at all. Within each group of the sensitive attribute, count how often the favourable outcome actually happened:

```
outcome rate(a) = (people in group a whose outcome is positive) / (people in group a)
```

`positive_label=1` names the value of `y` that counts as favourable — here, survival. The returned frame reports the group size `n` next to the rate, and the size matters as much as the rate: a proportion computed on a few dozen people moves by several points when one person changes side.

In this table women survived at 195/259 ≈ 75% and men at 93/453 ≈ 21%. That 54-point gap is **bias in the data**: `Survived` is strongly associated with `Sex` before anything is trained, and a learner allowed to read `Sex` has a large, real association available to copy.

Two things this number is not. It is not a prediction that the fitted model will be unfair — a model can leave an association in the data unused. And it is not, by itself, a verdict: whether a gap in the labels is historical record, measurement artefact, or discrimination to be undone is a question about how the data came to exist, not one the arithmetic can answer. What the rate does tell you is what the search is being offered.

In [2]:
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Titanic survival rate by sex')

    group    n  n_positive  positive_rate
0  female  259         195       0.752896
1    male  453          93       0.205298


<Axes: title={'center': 'Titanic survival rate by sex'}, xlabel='Group', ylabel='Positive rate'>

## Step 2. A fuzzy rule classifier, and how to read one

A fuzzy rule-based classifier is a list of IF–THEN sentences plus a way of choosing between them. It is explainable *by design*: the sentences printed below are not an explanation of the model, they **are** the model.

**Linguistic variables.** Each numeric column is divided into overlapping words — here three of them, `Low`, `Medium`, `High` (`n_linguistic_variables=3`). Membership is graded: a value belongs to `Low` to a degree between 0 and 1, so a passenger near a boundary is partly `Low` and partly `Medium` instead of flipping at a cutoff. `FUZZY_SETS.t1` asks for Type-1 sets, where that degree is a single number; the library also offers interval Type-2 sets, which put a band of uncertainty around it. Categorical columns stay categorical, so a condition reads `Sex IS female` or `Embarked IS C` and matches at 1 or 0.

**Firing strength.** The conditions of a rule are combined by multiplication:

```
firing strength = membership of condition 1 × membership of condition 2 × ...
```

If `Sex IS female` matches at 1 and `Age IS Medium` at 0.5, the rule fires at 0.5; any condition matching at 0 silences the rule entirely. `nAnts=3` caps a rule at three conditions, and slots may go unused, which is why printed rules are often shorter than three.

**The winning rule.** Every rule is scored for every passenger and the highest-scoring rule decides — a single winner, not a vote among rules of the same class. The score is the firing strength times the rule weight, and `ds_mode=1` fixes every weight at 1, so firing alone decides here. (`ds_mode=0` weights each rule by its dominance score; `ds_mode=2`, used in demo 11, lets the genetic search choose a weight per rule.) One winner per passenger is what makes "which rule fired" a complete account of a prediction.

**Where the sentences come from.** They are not written by hand. A genetic search keeps a population of candidate rule bases; each candidate picks conditions, words and consequents, and — since no `linguistic_variables` are passed here — also moves the boundaries of Low, Medium and High. Candidates are scored on the training rows by the Matthews correlation coefficient

```
MCC = (TP·TN − FP·FN) / sqrt( (TP+FP)(TP+FN)(TN+FP)(TN+FN) )
```

where TP and TN are the correct predictions of each class and FP and FN the two kinds of mistake. MCC runs from −1 (perfect disagreement) through 0 (no information) to +1, and unlike accuracy it is not flattered by a lazy majority rule on an uneven table. Note what is absent from that objective: nothing in it mentions groups or fairness. Demo 11 is where that changes.

**This is a classroom budget.** `n_gen=6`, `pop_size=12`, `patience=3` means twelve candidates, at most six generations, stopping after three generations without improvement. The library defaults are far larger. Read the stored rules as a demonstration of the measurements, not as the best rules these data admit — a longer search writes different sentences, but you read them the same way.

**Reading the printout.** `eval_fuzzy_model(..., print_rules=True, return_rules=True)` prints train and test accuracy and MCC, then the rule base in blocks, one block per consequent (0 = died, 1 = survived), and returns that text so the last cell can reuse it. Three numbers follow each rule:

- **DS**, the dominance score: how broadly the rule fires on its own class, set against how much it fires on the other. A large DS is a sentence the model leans on; a DS near zero is a rule that almost never wins. Rules whose score falls below `tolerance` are pruned during the search, which is one reason the printed list can be shorter than `nRules=8`.
- **ACC**: the share of cases in which this rule won *and* was right. A rare rule can show `ACC 1.0` and still decide nobody, so always read it next to DS.
- **WGHT**: the weight used at decision time, fixed at 1.0 by `ds_mode=1`.

When the rules appear, look for any that name `Sex` — and then look at their DS before concluding anything.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
clf = BaseFuzzyRulesClassifier(
    nRules=8, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
    n_linguistic_variables=3, ds_mode=1, verbose=False,
    n_gen=6, pop_size=12, patience=3, random_state=42,
)
clf.fit(X_train, y_train)
report = eval_tools.eval_fuzzy_model(
    clf, X_train, y_train, X_test, y_test,
    plot_rules=False, print_rules=True, plot_partitions=False,
    return_rules=True, bootstrap_results_print=False,
)
print(report[:1500] if report else '(no rule text)')

------------
ACCURACY
Train performance: 0.7526205450733753
Test performance: 0.6510638297872341
------------
MATTHEW CORRCOEF
Train performance: 0.4778647211133203
Test performance: 0.26215313062999657
------------
Rules for consequent: 0
----------------
IF Age IS Low AND Parch IS Medium AND Fare IS Medium WITH DS 0.000193219275412515, ACC 0.7853658536585366, WGHT 1.0
IF Sex IS female AND SibSp IS High WITH DS 0.0066384034664147195, ACC 0.8, WGHT 1.0
IF Parch IS Medium AND Fare IS Low AND Embarked IS C WITH DS 0.0001595028564434811, ACC 0.5, WGHT 1.0
IF Age IS Low AND SibSp IS Low WITH DS 0.04170820421016782, ACC 0.6506849315068494, WGHT 1.0
IF Parch IS High AND Fare IS Low WITH DS 0.006708595387840671, ACC 0.8, WGHT 1.0

Rules for consequent: 1
----------------
IF Sex IS female AND Age IS Medium WITH DS 0.10796843507156857, ACC 0.8256880733944955, WGHT 1.0


Rules for consequent: 0
----------------
IF Age IS Low AND Parch IS Medium AND Fare IS Medium WITH DS 0.000193219275412515, AC

## Steps 3 and 4. Bias in inference: rates, errors, and which rule fired

The same counting as in step 1, now applied to predictions instead of labels.

**Selection rate** is the outcome rate's mirror image: among people in a group, how often does the model *predict* the favourable outcome? `fairness_report` returns one row per group — selection rate plus the four confusion rates — and a dictionary of gaps:

- **TPR**, of those who truly survived, the share predicted to survive (recall within the group);
- **FPR**, of those who truly died, the share predicted to survive;
- **FNR** and **TNR**, their complements (`fnr = 1 − tpr`, `tnr = 1 − fpr`).

**Demographic parity** compares selection rates only: `difference = max − min` across groups (0 when equal) and `ratio = min / max` (1 when equal). It asks whether the groups are selected at the same rate and deliberately ignores whether those selections were correct.

**Equalized odds** compares error rates instead. The printed `equalized_odds_difference` is the *larger* of the TPR gap and the FPR gap, and the ratio is the smaller of the two rate ratios. It asks whether the model is equally right and equally wrong in each group, which is the criterion to prefer when the recorded label is itself trustworthy.

The two criteria disagree whenever the base rates differ, and that disagreement is the substance of the subject rather than a technicality. If the outcome is genuinely more common in one group, a model with perfect equalized odds will show unequal selection rates; forcing equal selection rates then means predicting some people wrongly on purpose. Only when the base rates are equal can both hold at once. Choosing between them is a decision about the world, not about the code — which is why `ebdai` prints both and picks neither.

Two reading traps in the numbers below:

- they are proportions, so `0.57` means **57 percentage points**, not 0.57%;
- a ratio of 0 almost always means one group's rate is exactly 0, which may rest on a handful of people rather than a deep separation. Read `tpr_difference` and `fpr_difference`, with the group sizes, before reading any ratio.

**Which rule fired.** `explainable_predict` returns the prediction together with the index of the winning rule, its association degree and a confidence interval. `winning_rules_by_group` takes those winner indexes and counts them inside each group, as counts and as a share of the group; `parse_printed_rules` turns the printed report into one string per rule in the same order as the indexes, so the chart is labelled with the sentences themselves.

This is the step a black box cannot offer. The fairness table tells you *that* the groups are treated differently; the winning-rule table tells you *which sentence does it*. In the stored run they meet: a single rule, `IF Sex IS female AND Age IS Medium THEN 1`, is the only source of predicted survivals, so the selection rate for men is exactly 0 — every man in the test set is predicted to have died, including the 35 who survived.

In [4]:
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print(table)
print(pd.Series(gaps))

texts = parse_printed_rules(report or '')
counts = winning_rules_by_group(clf, X_test, X_test[sensitive], rule_texts=texts)
print(counts.head())
plot_winning_rules_by_group(counts, title='Winning Titanic rules by sex')

    group    n  selection_rate       tpr       fpr       fnr       tnr
0  female   86        0.569767  0.515625  0.727273  0.484375  0.272727
1    male  149        0.000000  0.000000  0.000000  1.000000  1.000000
demographic_parity_difference    0.569767
demographic_parity_ratio         0.000000
equalized_odds_difference        0.727273
equalized_odds_ratio             0.000000
tpr_difference                   0.515625
fpr_difference                   0.727273
dtype: float64
    group  rule  count      rate  \
0  female     0      9  0.104651   
1  female     1      1  0.011628   
2  female     2      2  0.023256   
3  female     3     24  0.279070   
4  female     4      1  0.011628   

                                           rule_text  
0  IF Age IS Low AND Parch IS Medium AND Fare IS ...  
1          IF Sex IS female AND SibSp IS High THEN 0  
2  IF Parch IS Medium AND Fare IS Low AND Embarke...  
3              IF Age IS Low AND SibSp IS Low THEN 0  
4            IF Parch IS Hig

<Axes: title={'center': 'Winning Titanic rules by sex'}, xlabel='Winning rule', ylabel='Share of group'>